In [ ]:
from database.db import SessionLocal
from database.models import Employee, Investigation

session = SessionLocal()
employees = session.query(Employee).all()
for emp in employees:
    print(emp.id, emp.name, emp.policy_limit, emp.risk_tier, emp.manager_slack_id)
print(Investigation)
session.close()

[]


In [9]:
from database.models import ExpensePolicy
policy = session.query(ExpensePolicy).all()
for p in policy:
    print(p.category, p.max_amount, p.requires_receipt, p.note)
session.close()


Office Supplies 150.0 False Standard supplies
Meals 50.0 True Client meals require receipt
Travel 1000.0 True Airfare and hotel


In [10]:
from database.db import engine
from sqlalchemy import text

def run_raw_sql(query: str):
    """Executes a raw SQL query and prints results."""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        for row in result:
            print(row)
    conn.close()

# Example usage:

run_raw_sql("SELECT * FROM transactions-- where risk_tier = 'low'")

('T100', 'E456', '2026-05-04 10:22:45.243388', 'Office Supplies', 120.0, 'Staples', 'completed', 0)
('T101', 'E456', '2026-05-09 10:22:45.243388', 'Meals', 45.0, 'Chipotle', 'completed', 0)
('T102', 'E456', '2026-05-14 10:22:45.243388', 'Office Supplies', 145.0, 'Office Depot', 'completed', 0)
('T103', 'E456', '2026-05-19 10:22:45.243388', 'Office Supplies', 148.0, 'Amazon Business', 'completed', 0)
('T104', 'E456', '2026-05-24 10:22:45.244386', 'Travel', 850.0, 'United Airlines', 'flagged', 0)
('T105', 'E456', '2026-05-29 10:22:45.244386', 'Office Supplies', 147.0, 'Staples', 'completed', 0)
('T106', 'E789', '2026-05-31 10:22:45.244386', 'Meals', 55.0, 'Steakhouse', 'completed', 0)
('T107', 'E789', '2026-06-01 10:22:45.244386', 'Meals', 62.0, 'Seafood Place', 'completed', 0)


In [11]:
import json

def get_employee_expense_profile(employee_id: str):
    """Return employee profile as a JSON string for the LLM's observation."""
    session = SessionLocal()
    try:
        employee = session.query(Employee).filter(Employee.id == employee_id).first()
        # employee profile as a JSON string
        employee_profile = {"employee_id": employee.id,
            "name": employee.name,
            "policy_limit": employee.policy_limit,
            "risk_tier": employee.risk_tier,
            "manager_slack_id": employee.manager_slack_id}
        return json.dumps(employee_profile)

    except Exception as e:
        """Return an error string the LLM can understand"""
        return json.dumps({"error": f"Database error: {str(e)}"})

    finally:
        session.close()

get_employee_expense_profile("E456")

'{"employee_id": "E456", "name": "Alice", "policy_limit": 200.0, "risk_tier": "low", "manager_slack_id": "U123"}'

In [20]:
import datetime
from zoneinfo import ZoneInfo

# Get the current time explicitly in IST
today = datetime.datetime.today().date()

print(today)

2026-06-03


In [ ]:

import os
import requests
from dotenv import load_dotenv

load_dotenv()

# for Telegram
telegram_bot_token = os.getenv("TELEGRAM_TOKEN")
telegram_chatid = os.getenv("TELEGRAM_CHATID")

message = "Hii Dude! "

def send_telegram_escalation(message):
    url = f"https://api.telegram.org/bot{telegram_bot_token}/sendMessage"

    # escalation_message = create_escalation_message(employee_id, transaction_details, escalation_reason, evidence_summary, employee_slack_id)

    # Using MarkdownV2 or HTML is much more stable for multi-agent variables
    payload = {"chat_id": telegram_chatid, "text": message, "parse_mode": "Markdown"}

    try:
        # Always use json= payload for streaming LLM text to avoid form-encoding corruption
        response = requests.post(url, json=payload)
        response.raise_for_status() 
        return {"status": "Success", "platform": "Telegram"}
    except Exception as e:
        if 'response' in locals() and response is not None:
            return {"status": "Error", "message": f"Telegram Rejected Payload: {response.text}"}
        return {"status": "Error", "message": str(e)}

f = send_telegram_escalation(message)
print(f)

{'status': 'Success', 'platform': 'Telegram'}
